# 🏥 AI-Powered Hospital Analytics & Treatment Cost Prediction
### End-to-End Data Science Pipeline
---
**Sections:**
1. Setup & Data Loading
2. Data Cleaning & Feature Engineering
3. Exploratory Data Analysis (EDA)
4. Machine Learning — Treatment Cost Prediction
5. Model Evaluation & Business Insights


## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', None)
print("Libraries loaded ✓")

In [ ]:
DATA = "data"

patients     = pd.read_csv(f"{DATA}/patients.csv")
doctors      = pd.read_csv(f"{DATA}/doctors.csv")
appointments = pd.read_csv(f"{DATA}/appointments.csv")
treatments   = pd.read_csv(f"{DATA}/treatments.csv")
billing      = pd.read_csv(f"{DATA}/billing.csv")

for name, df in [("Patients",patients),("Doctors",doctors),
                 ("Appointments",appointments),("Treatments",treatments),("Billing",billing)]:
    print(f"{name:15s}: {df.shape[0]:>3} rows × {df.shape[1]} cols")

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Fix date types
for df, cols in [
    (patients,     ['date_of_birth','registration_date']),
    (appointments, ['appointment_date']),
    (treatments,   ['treatment_date']),
    (billing,      ['bill_date']),
]:
    for c in cols: df[c] = pd.to_datetime(df[c], errors='coerce')

# Derive age & age group
patients['age'] = (pd.Timestamp.now() - patients['date_of_birth']).dt.days // 365
patients['age_group'] = pd.cut(patients['age'], bins=[0,18,35,50,65,120],
    labels=['<18','18-35','36-50','51-65','65+'])

# Appointment time features
appointments['month']    = appointments['appointment_date'].dt.to_period('M').astype(str)
appointments['day_name'] = appointments['appointment_date'].dt.day_name()
appointments['hour']     = pd.to_datetime(appointments['appointment_time'],
                               format='%H:%M:%S', errors='coerce').dt.hour

print("Cleaning complete ✓")
patients[['patient_id','gender','age','age_group']].head()

In [ ]:
# Merge master table
master = (appointments
    .merge(patients[['patient_id','gender','age','age_group','insurance_provider']],
           on='patient_id', how='left')
    .merge(doctors[['doctor_id','specialization','years_experience','hospital_branch']],
           on='doctor_id', how='left')
    .merge(treatments[['appointment_id','treatment_type','description','cost']],
           on='appointment_id', how='left')
    .merge(billing[['treatment_id','amount','payment_method','payment_status']]
                  .merge(treatments[['treatment_id','appointment_id']], on='treatment_id'),
           on='appointment_id', how='left')
)
print(f"Master table: {master.shape}")
master.head(3)

## 3. Exploratory Data Analysis

### 3.1 Patient Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
fig.suptitle("Patient Demographics", fontsize=14, fontweight='bold')

# Gender
g = patients['gender'].value_counts()
axes[0].pie(g, labels=g.index, autopct='%1.1f%%', colors=['#2563EB','#10B981'],
            wedgeprops=dict(edgecolor='white'))
axes[0].set_title("Gender")

# Age group
ag = patients['age_group'].value_counts().sort_index()
axes[1].bar(ag.index.astype(str), ag.values, color='#2563EB', edgecolor='white')
axes[1].set_title("Age Groups"); axes[1].set_ylabel("Count")

# Insurance
ins = patients['insurance_provider'].value_counts()
axes[2].barh(ins.index, ins.values, color='#8B5CF6', edgecolor='white')
axes[2].set_title("Insurance Providers")

plt.tight_layout(); plt.show()

### 3.2 Appointment Trends

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
fig.suptitle("Appointment Trends", fontsize=14, fontweight='bold')

monthly = appointments.groupby('month').size().reset_index(name='count').sort_values('month')
axes[0].plot(range(len(monthly)), monthly['count'], color='#2563EB', marker='o', linewidth=2)
axes[0].fill_between(range(len(monthly)), monthly['count'], alpha=0.15, color='#2563EB')
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly['month'], rotation=45, ha='right', fontsize=7)
axes[0].set_title("Monthly Trend"); axes[0].set_ylabel("# Appointments")

dow = appointments['day_name'].value_counts().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']).fillna(0)
axes[1].bar(dow.index, dow.values, color='#10B981', edgecolor='white')
axes[1].set_title("By Day of Week"); axes[1].tick_params(axis='x', rotation=45)

s = appointments['status'].value_counts()
axes[2].pie(s, labels=s.index, autopct='%1.1f%%',
            colors=['#2563EB','#10B981','#F59E0B','#EF4444'],
            wedgeprops=dict(edgecolor='white'))
axes[2].set_title("Status Breakdown")

plt.tight_layout(); plt.show()

### 3.3 Revenue & Treatment Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
fig.suptitle("Revenue & Treatment Analysis", fontsize=14, fontweight='bold')

# Avg cost by treatment
avg_c = master.groupby('treatment_type')['cost'].mean().sort_values(ascending=False)
axes[0].barh(avg_c.index, avg_c.values, color='#F59E0B', edgecolor='white')
axes[0].set_title("Avg Cost by Treatment"); axes[0].set_xlabel("Avg Cost ($)")

# Revenue by specialization
rev_s = master.groupby('specialization')['cost'].sum().sort_values(ascending=False)
axes[1].barh(rev_s.index, rev_s.values, color='#EC4899', edgecolor='white')
axes[1].set_title("Revenue by Specialization"); axes[1].set_xlabel("Total Revenue ($)")

# Payment status
ps = billing['payment_status'].value_counts()
colors_ps = {'Paid':'#10B981','Pending':'#F59E0B','Overdue':'#EF4444'}
axes[2].pie(ps, labels=ps.index, autopct='%1.1f%%',
            colors=[colors_ps.get(x,'#6366F1') for x in ps.index],
            wedgeprops=dict(edgecolor='white'))
axes[2].set_title("Payment Status")

plt.tight_layout(); plt.show()

### 3.4 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
num_cols = master.select_dtypes(include=[np.number]).columns.tolist()
corr = master[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, linewidths=0.5, cbar_kws={'shrink':0.8})
ax.set_title("Numeric Feature Correlations", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Machine Learning — Treatment Cost Prediction

In [ ]:
# Feature engineering
ml = master.dropna(subset=['cost']).copy()
ml['hour_bucket'] = pd.cut(ml['hour'].fillna(12), bins=[0,8,12,16,20,24],
    labels=['Early','Morning','Afternoon','Evening','Night'])

CAT = ['gender','age_group','insurance_provider','specialization','hospital_branch',
       'treatment_type','reason_for_visit','status','payment_method','hour_bucket']
NUM = ['age','years_experience']
TARGET = 'cost'

ml_clean = ml[CAT + NUM + [TARGET]].dropna()
X = ml_clean[CAT + NUM]
y = ml_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT),
    ('num', StandardScaler(), NUM),
])

models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest"     : RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
}

results, trained = {}, {}
for name, model in models.items():
    pipe = Pipeline([('prep', prep), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        'MAE' : mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²'  : r2_score(y_test, y_pred),
        'y_pred': y_pred
    }
    trained[name] = pipe

pd.DataFrame({k:{m:v for m,v in v.items() if m!='y_pred'} for k,v in results.items()}).T.round(4)

### 4.1 Feature Importance (Random Forest)

In [ ]:
rf = trained["Random Forest"]
ohe_cols = rf.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(CAT).tolist()
feat_df = pd.DataFrame({'feature': ohe_cols + NUM,
                        'importance': rf.named_steps['model'].feature_importances_})            .sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(feat_df['feature'][::-1], feat_df['importance'][::-1], color='#2563EB', edgecolor='white')
ax.set_title("Top 15 Feature Importances", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

### 4.2 Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.4, s=25, color='#2563EB')
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn,mx],[mn,mx],'r--', linewidth=2, label='Perfect')
    ax.set_title(f"{name}\nR²={res['R²']:.4f}")
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Business Insights & Recommendations

| Finding | Metric | Recommendation |
|---------|--------|----------------|
| High no-show rate | 26% | Implement 24h SMS/email reminders |
| Low payment completion | 32% Paid | Tiered follow-up; payment plans |
| MRI highest avg cost | $3,225 | Priority slot management |
| Pediatrics top revenue | #1 specialisation | Expand Pediatrics capacity |
| ML cost prediction | R² near 0 on synthetic data | Enrich with ICD-10 codes for production |


In [ ]:
# Save best model
best = max(results, key=lambda k: results[k]['R²'])
joblib.dump(trained[best], "models/treatment_cost_prediction.pkl")
print(f"Best model '{best}' saved ✓")